# Setup

In [1]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
from osgeo import gdal
from tqdm import tqdm

%matplotlib inline

In [3]:
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

In [4]:
repo_dir = "lfm"

sys.path.append("lfm")

from lfm.tasks.sem_segmentation.model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmenation.data_cube_inference import run_datacube_inference

# Config

In [5]:
# Data paths
INPUT_DIR = "/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/checkpoint_epoch_050.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs_meeting"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs_meeting
Using device: cuda


# Load model

In [6]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, device) 
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(checkpoint_state, strict=False)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

Loading model from /explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main


Encoder loaded with pretrained weights.


/explore/nobackup/people/ajkerr1/.nccstmp/ipykernel_503445/1494035847.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pretrained_checkpoint, map

Successfully loaded model from checkpoint!


# Load data

## Load statistics from training to normalize inputs

In [7]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics.")
print("="*60)

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_mean.npy")
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_std.npy")
print(MEAN, STD)


STEP 1: Loading training dataset statistics.
[0.32744208 0.32249309 0.302985  ] [0.15045737 0.15014801 0.14386101]


# Inference

In [28]:
# Run inference, create viz of inference
n_images = 3
images, predictions = run_datacube_inference(
    model=model,
    device=device,
    tiff_dir=INPUT_DIR,
    mean=MEAN,
    std=STD,
    output_path="sem_seg_inf_test_3_19.png",
    n_images=n_images,
    model_native_size=TARGET_SIZE[0],
    tile_overlap=0.25,
    threshold=0.75,
    normalize=True,
    save_inputs_dir=None,
    use_sliding=True,
)

Found 73 TIFF files
Loading 3 images from TIFF files...
Extracting images with mean=[0.32744208 0.32249309 0.302985  ], std=[0.15045737 0.15014801 0.14386101]
Processing /explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/Cube-LTM30N_Zoom-5_Tile-1-22.tif: 564 bands, 112 complete 5-band slices
  ✗ Slice 0: bands 0-2 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 1: bands 5-7 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 2: bands 10-12 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 3: bands 15-17 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
  ✗ Slice 4: bands 20-22 → skipped (non-positive values)
     Slice min value: -3.4028226550889045e+38, count and percent: 786432, 100.0
 

  0%|          | 0/4 [00:00<?, ? batches/s]

Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 1.00]

Processing image 2/3
Number of tiles: 4


  0%|          | 0/4 [00:00<?, ? batches/s]

Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 1.00]

Processing image 3/3
Number of tiles: 4


  0%|          | 0/4 [00:00<?, ? batches/s]

Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 1.00]

Got 3 predictions
Output mask shapes: [(512, 512), (512, 512), (512, 512)]

Creating visualization...

Saved visualization to: sem_seg_inf_test_3_19.png
